3D graphics for Lecture 2
=========================

**Author:** Marcos Bujosa



$
\newcommand{\R}[1][{}]{{\mathbb{R}}^{#1}}
\newcommand{\Z}[1][{}]{{\mathbb{Z}}^{#1}}
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
\newcommand{\getItem}{{\mid}}
\newcommand{\getitemR}[1]{\getItem{#1}}
\newcommand{\getitemL}[1]{{#1}\getItem}
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
\newcommand{\elemR}[2]{{#1}^{\phantom{\T}}_{\getitemR{#2}}} 
\newcommand{\elemRP}[2]{{\big(#1\big)}^{\phantom{\T}}_{\getitemR{#2}}} 
\newcommand{\elemRPE}[2]{\big({#1}^{\phantom{\T}}_{\getitemR{#2}}\big)}
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
\newcommand{\elemL}[2]  { {\vphantom{#1}}_{ \getitemL{#2}} {#1} } 
\newcommand{\elemLP}[2] { \elemL{\!\big({#1}\big)}{#2} }  
\newcommand{\elemLPE}[2]{ \big(\elemL{#1}{#2}\big) }
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
\newcommand{\Vect}[2][{}]{{\boldsymbol{#2}}_{#1}}
\newcommand{\eleVR}[2]  {\elemR  {\Vect{#1}}{#2}}      % con subindices
\newcommand{\eleVRP}[2] {\elemRP {\Vect{#1}}{#2}}      % con subindices y paréntesis interior
\newcommand{\eleVRPE}[2]{\elemRPE{\Vect{#1}}{#2}}      % con subindices y paréntesis exterior
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
\newcommand{\VectC}[2]  {\elemR  {\Mat{#1}}{#2}}         % con subindices
\newcommand{\VectCP}[2] {\elemRP {\Mat{#1}}{#2}}       % con subindices y paréntesis
\newcommand{\VectCPE}[2]{\elemRPE{\Mat{#1}}{#2}}       % con subindices y paréntesis exterior
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
\newcommand{\VectF}[2]  {\elemL  {\Mat{#1}}{#2}}         % con subindices
\newcommand{\VectFP}[2] {\elemLP {\Mat{#1}}{#2}}       % con subindices y paréntesis
\newcommand{\VectFPE}[2]{\elemLPE{\Mat{#1}}{#2}}       % con subindices y paréntesis exterior
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
\newcommand{\mat}[1]{\mathbf{#1}}
%\newcommand{\mat}[1]{\boldsymbol{#1}}                 % hace cursiva, no me gusta para matriz
\newcommand{\Mat} [2][{}]{{\mat{#2}}_{#1}} 
\newcommand{\T}{\intercal}
\newcommand{\MatT}[2][{}]{{\mat{#2}}^{\T}_{#1}}
\newcommand{\VectCC}[2][{}]   {\elemRR {\Mat{#2}}{#1}}   % con ()
\newcommand{\VectCCC}[2][{}]  {\elemRRR{\Mat{#2}}{#1}}   % con texto "col"
%SELECCIÓNA de FILAS y COlUMNAS DE UNA MATRIZ TRANSPUESTA PARA GENERAR UN VECTOR DE Rn
\newcommand{\VectTC}[2][{}]   {\elemR{\MatT{#2}\!}{#1}}  % con subindices
\newcommand{\VectTCC}[2][{}]  {\elemRR{ \MatT{#2}}{#1}}  % con ()
\newcommand{\VectTCCC}[2][{}] {\elemRRR{\MatT{#2}}{#1}}  % con texto "col"
\newcommand{\dotprod}[2][{}] {\Vect{#1}\cdot\Vect{#2}}
\newcommand{\concat}{\mathbin{-\mkern-9.5mu\lVert}}
\newcommand{\SEL}[2] { \Mat{#1}\Vect{x}=\Vect{#2} }
\newcommand{\EV}[1]{\mathcal{#1}}
\newcommand{\SV}[1]{\mathsf{#1}}
$

This Notebook does not accompany a section of the book. It draws in $\R[3]$ the examples of the slides for Lecture 2 of the class material: the linear combinations of two vectors and the two pictures of a system of three equations in three unknowns. The figures can be rotated. Do the [Notebook for Section 2.2](Lecc02-Secc2.2-Eng.ipynb) first. To draw in the plane use the Notebook [Drawing vectors in the plane](DrawingVectors-Eng.ipynb).

To use [NAcAL](https://mbujosab.github.io/nacallib/) you need to import the `nacal` module.



In [1]:
from nacal import *

## The drawing functions



NAcAL does not draw. The next cell defines three functions that draw in $\R[3]$: `draw_combinations`, `draw_rows` and `draw_columns`. They use the [Plotly](https://plotly.com/python/) library. You do not need to understand their code. Run the cell and go on to the next part.



In [1]:
import numpy as np
import plotly.graph_objects as go

COLOURS = ['royalblue', 'firebrick', 'seagreen']

def _numbers(v):
    if not isinstance(v, Vector) or v.n != 3:
        raise ValueError('A vector of R^3 is needed.')
    return np.array([float(i|v) for i in (1, 2, 3)])

def _table(A):
    if not isinstance(A, Matrix) or A.m != 3 or A.n != 3:
        raise ValueError('A matrix of order 3 by 3 is needed.')
    return np.array([[float(i|A|j) for j in (1, 2, 3)] for i in (1, 2, 3)])

def _text(p):
    return '(' + ', '.join('%g' % (round(float(c), 4) + 0.0) for c in p) + ',)'

def _arrow(fig, start, end, color, name, width=8, head=0.5):
    start = np.array(start, float)
    end = np.array(end, float)
    d = end - start
    length = np.linalg.norm(d)
    fig.add_trace(go.Scatter3d(x=[start[0], end[0]], y=[start[1], end[1]],
                               z=[start[2], end[2]], mode='lines',
                               line=dict(color=color, width=width),
                               name=name, legendgroup=name))
    if length < 1e-9:
        return
    d = d / length
    u = np.cross(d, [0, 0, 1])
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(d, [0, 1, 0])
    u = u / np.linalg.norm(u)
    w = np.cross(d, u)
    height = min(head, length / 2)
    base = end - height * d
    k = 12
    ring = [base + 0.3 * height * (np.cos(t) * u + np.sin(t) * w)
           for t in np.linspace(0, 2 * np.pi, k, endpoint=False)]
    p = np.array([end, base] + ring)
    fig.add_trace(go.Mesh3d(x=p[:, 0], y=p[:, 1], z=p[:, 2],
                            i=[0] * k + [1] * k,
                            j=[2 + n for n in range(k)] * 2,
                            k=[2 + (n + 1) % k for n in range(k)] * 2,
                            color=color, flatshading=True, hoverinfo='skip',
                            legendgroup=name, showlegend=False))

def _piece_of_plane(fig, centre, u, w, color, name):
    centre = np.array(centre, float)
    p = np.array([centre - u - w, centre + u - w, centre + u + w, centre - u + w])
    fig.add_trace(go.Mesh3d(x=p[:, 0], y=p[:, 1], z=p[:, 2],
                            i=[0, 0], j=[1, 2], k=[2, 3],
                            color=color, opacity=0.45, flatshading=True,
                            hoverinfo='skip', name=name, showlegend=True))
    edge = np.vstack([p, p[:1]])
    fig.add_trace(go.Scatter3d(x=edge[:, 0], y=edge[:, 1], z=edge[:, 2],
                               mode='lines', line=dict(color=color, width=2),
                               hoverinfo='skip', showlegend=False))

def _endish(fig, title, figure):
    fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[0], mode='markers',
                               marker=dict(color='black', size=4),
                               name='origin'))
    fig.update_layout(title=title, height=650,
                      margin=dict(l=0, r=0, t=50, b=0),
                      scene=dict(aspectmode='data', xaxis_title='x',
                                 yaxis_title='y', zaxis_title='z',
                                 camera=dict(eye=dict(x=2.0, y=0.3, z=0.9))))
    if figure:
        return fig
    fig.show()

def draw_combinations(v, w, x=1, y=1, figure=False):
    """Draws two vectors of R^3, the piece of plane filled by their
    linear combinations and the linear combination x*v + y*w."""
    va, vb = _numbers(v), _numbers(w)
    x, y = float(x), float(y)
    fig = go.Figure()
    reach = max(1.5, abs(x) + 0.5, abs(y) + 0.5)
    _piece_of_plane(fig, [0, 0, 0], reach * va, reach * vb, 'goldenrod',
                    'linear combinations of v and w')
    head = 0.06 * max(np.linalg.norm(va), np.linalg.norm(vb), 1)
    _arrow(fig, [0, 0, 0], va, COLOURS[0], 'v = ' + _text(va), head=head)
    _arrow(fig, [0, 0, 0], vb, COLOURS[1], 'w = ' + _text(vb), head=head)
    total = x * va + y * vb
    _arrow(fig, x * va, total, 'lightcoral', '%g w, from the tip of %g v' % (y, x),
            width=4, head=head)
    _arrow(fig, [0, 0, 0], total, 'black',
            '%g v %s %g w = ' % (x, '+' if y >= 0 else '-', abs(y)) + _text(total), width=5, head=head)
    return _endish(fig, 'Linear combinations of two vectors', figure)

def draw_rows(A, b, figure=False):
    """Row picture of the system Ax = b: draws the plane of each
    equation and, if the solution is unique, the point where they meet."""
    M, rhs = _table(A), _numbers(b)
    fig = go.Figure()
    solution = np.linalg.solve(M, rhs) if abs(np.linalg.det(M)) > 1e-9 else None
    size = 3 if solution is None else 3 + np.linalg.norm(solution) / 2
    for i in range(3):
        n = M[i]
        if np.linalg.norm(n) < 1e-9:
            print('Equation', i + 1, 'has no unknowns: it is not a plane.')
            continue
        n1 = n / np.linalg.norm(n)
        centre = rhs[i] * n / n.dot(n)
        if solution is not None:
            centre = solution
        u = np.cross(n1, [0, 0, 1])
        if np.linalg.norm(u) < 1e-6:
            u = np.cross(n1, [0, 1, 0])
        u = u / np.linalg.norm(u)
        w = np.cross(n1, u)
        _piece_of_plane(fig, centre, size * u, size * w, COLOURS[i],
                        'equation %d' % (i + 1))
    if solution is None:
        title = 'Row picture: the three planes do not meet in a single point'
    else:
        fig.add_trace(go.Scatter3d(x=[solution[0]], y=[solution[1]], z=[solution[2]],
                                   mode='markers', marker=dict(color='black', size=7),
                                   name='solution ' + _text(solution)))
        title = 'Row picture: the three planes meet at ' + _text(solution)
    return _endish(fig, title, figure)

def draw_columns(A, b, figure=False):
    """Column picture of the system Ax = b: draws the columns of A,
    the vector b and, if the solution is unique, the linear combination
    of the columns that gives b."""
    M, rhs = _table(A), _numbers(b)
    fig = go.Figure()
    head = 0.06 * max(np.linalg.norm(rhs), max(np.linalg.norm(M[:, j]) for j in range(3)), 1)
    _arrow(fig, [0, 0, 0], rhs, 'black', 'b = ' + _text(rhs), width=12, head=head)
    for j in range(3):
        _arrow(fig, [0, 0, 0], M[:, j], COLOURS[j],
                'column %d = ' % (j + 1) + _text(M[:, j]), head=head)
    if abs(np.linalg.det(M)) > 1e-9:
        solution = np.linalg.solve(M, rhs)
        start = np.zeros(3)
        for j in range(3):
            end = start + solution[j] * M[:, j]
            _arrow(fig, start, end, COLOURS[j],
                    '%g times column %d' % (round(solution[j], 4) + 0.0, j + 1),
                    width=4, head=head)
            start = end
        title = 'Column picture: b is the linear combination with coefficients ' + _text(solution)
    else:
        title = 'Column picture: there is not a single linear combination that gives b'
    return _endish(fig, title, figure)

## Linear combinations of two vectors



The next cell draws two vectors of $\R[3]$ as arrows from the origin. It also draws a piece of the plane filled by their linear combinations.



In [1]:
v = Vector([4, 6, 10])
w = Vector([-4, 8, 4])
draw_combinations(v, w)

The figure is moved with the mouse:

-   Drag with the left button to rotate it.
-   Use the wheel to zoom in or out.
-   Click a name in the legend to hide or show that element.
-   The button with the shape of a house, at the top right, goes back to the initial view.

Rotate the figure until you see the yellow piece edge-on. It looks like a segment, because it is flat. The arrows of $\Vect{v}$ and $\Vect{w}$ are inside it.

The black arrow is the linear combination $\Vect{v}+\Vect{w}$. The pink arrow is $\Vect{w}$ placed after $\Vect{v}$. The last two arguments of the function are the coefficients of the combination. The next cell draws $2\Vect{v}-\Vect{w}$.



In [1]:
draw_combinations(v, w, 2, -1)

Draw other linear combinations of $\Vect{v}$ and $\Vect{w}$, with the coefficients you want. Rotate each figure. Does the black arrow ever leave the yellow plane?



The figure only shows a piece. The set of all linear combinations of $\Vect{v}$ and $\Vect{w}$ is the whole plane, which passes through the origin.

Now draw the linear combinations of $\Vect{v}$ and $2\Vect{v}$. What do they fill?



## The row picture of a system of equations



The following system is the one in the slides for Lecture 2. It has three equations and three unknowns.



In [1]:
A = Matrix([ [2,-1,0], [-1,2,-1], [0,-3,4] ])
b = Vector([0, -1, 4])
A

Each row of $\Mat{A}$ gives an equation. The first one is $2x-y=0$. The points of $\R[3]$ that satisfy an equation form a plane. The next cell draws a piece of each of the three planes.



In [1]:
draw_rows(A, b)

Rotate the figure. Two planes meet in a line. The three planes have a single point in common, which is the solution of the system. The title of the figure says which one it is. Check that $\Mat{A}$ times that vector gives $\Vect{b}$.



The slides then change the right-hand side to the following vector. Draw the row picture of the new system. Which planes have moved? Check the solution given in the title.



In [1]:
c = Vector([1, 1, -3])

## The column picture of the same system



To solve $\Mat{A}\Vect{x}=\Vect{b}$ is to look for the linear combination of the columns of $\Mat{A}$ that gives $\Vect{b}$. It is what you saw in the Notebook for Section 2.2. The next cell draws the three columns of $\Mat{A}$ and the vector $\Vect{c}$, which is the black arrow.



In [1]:
draw_columns(A, c)

The thin arrows are the linear combination: each column multiplied by its coefficient and placed after the previous one. The last one ends at the tip of the black arrow. The coefficients are the solution you saw in the row picture. Here they are 1, 1 and 0: the vector $\Vect{c}$ is the sum of the first two columns and the third one is not needed.

Draw the column picture with the vector $\Vect{b}$. Why does the black arrow hide one of the columns?



Choose another right-hand side yourself and draw the two pictures. Then check with the product that the coefficients in the title give your vector.



## When the three columns are in the same plane



The third column of the following matrix is the sum of the other two.



In [1]:
S = Matrix([ [2,-1,1], [-1,2,1], [0,-3,-3] ])
draw_columns(S, b)

Rotate the figure until you see the three columns edge-on. They are in the same plane and the black arrow is outside it. No linear combination of the columns gives $\Vect{b}$: the system has no solution.

Draw the row picture of that system. See whether the three planes have any point in common.



Now draw the two pictures with the matrix `S` and the vector $\Vect{c}$. That vector is in the plane of the columns.



## Python details



-   The three functions only accept vectors of $\R[3]$ and matrices of order 3 by 3.
-   `draw_combinations(v, w, x, y)` draws the linear combination with coefficients `x` and `y`. Without them it uses 1 and 1.
-   The functions compute the solution with decimal numbers in order to draw it. That is why the title may show a rounded value.
-   With the argument `figure=True` the function does not draw and returns the Plotly figure.



## What you should know at the end



-   The linear combinations of two vectors of $\R[3]$ that are not multiples of each other fill a plane through the origin.
-   In the row picture each equation is a plane. The solution is the point common to the three planes.
-   In the column picture the solution gives the coefficients of the linear combination of the columns that is equal to the right-hand side.
-   If the three columns are in the same plane, only the systems whose right-hand side is in that plane have a solution.

